In [2]:
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Use Microsoft Entra ID (Azure AD) for authentication
# Requires: az login, or a managed identity / service principal
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = AzureOpenAI(
    azure_endpoint="https://discovery-eastus2.openai.azure.com",   # e.g. https://<resource>.openai.azure.com
    azure_ad_token_provider=token_provider,
    api_version="2025-04-01-preview",                      # version that supports Responses API + code_interpreter
)

container = client.containers.create(name="test-container", memory_limit="4g")

response = client.responses.create(
    model="gpt-4.1-mini",           # your gpt-4.1 deployment name
    tools=[{
        "type": "code_interpreter",
        "container": container.id
    }],
    tool_choice="required",
    input="use the python tool to calculate what is 4 * 3.82. and then find its square root and then find the square root of that result"
)

print(response.output_text)


The calculations are as follows:

- 4 * 3.82 = 15.28
- The square root of 15.28 is approximately 3.91
- The square root of 3.91 is approximately 1.98


In [4]:
from agent_framework import ChatAgent, HostedCodeInterpreterTool
from agent_framework.openai import OpenAIChatClient
from azure.identity.aio import DefaultAzureCredential

async def main():
    # Use Microsoft Entra ID (Azure AD) for authentication
    # Requires: az login, or a managed identity / service principal
    async with DefaultAzureCredential() as credential:
        # OpenAIChatClient routes to Azure when azure_endpoint + credential are provided
        client = OpenAIChatClient(
            model="gpt-4.1-mini",  # Azure OpenAI deployment name
            azure_endpoint="https://discovery-eastus2.openai.azure.com",
            credential=credential,
            api_version="2025-04-01-preview",
        )

        # Create agent with sandboxed code execution capability
        agent = ChatAgent(
            chat_client=client,
            name="CodeAnalystAgent",
            instructions=(
                "You can execute Python code in a sandboxed environment. "
                "Use the code_interpreter tool to analyze data, create visualizations, "
                "or perform calculations. All code runs safely in an isolated sandbox."
            ),
            tools=[HostedCodeInterpreterTool()],  # Built-in hosted Code Interpreter
        )

        # Example: Data analysis using hosted Code Interpreter
        query = (
            "I have this data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]. "
            "Please calculate the mean, median, and standard deviation."
        )

        result = await agent.run(query)
        print(result)
        # Code executes in Azure OpenAI sandbox, not locally

# In a notebook, use `await main()` instead of `asyncio.run(main())`
await main()


ImportError: cannot import name 'ChatAgent' from 'agent_framework' (d:\vs_project\claude-code-analysis\.venv\Lib\site-packages\agent_framework\__init__.py)

In [5]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

async def main():
    # Create Azure OpenAI client with Code Interpreter support
    client = OpenAIChatClient(
        api_key="your-api-key",
        model="gpt-4-turbo"
    )
    
    # Create agent with sandboxed code execution capability
    agent = Agent(
        client=client,
        name="CodeAnalystAgent",
        instructions="""You can execute Python code in a sandboxed environment.
        Use the code_interpreter tool to analyze data, create visualizations, 
        or perform calculations. All code runs safely in an isolated sandbox.""",
        tools=["code_interpreter"]  # Built-in hosted Code Interpreter
    )
    
    # Example: Data analysis using hosted Code Interpreter
    query = """
    I have this data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    Please calculate the mean, median, and standard deviation, 
    then create a simple visualization.
    """
    
    result = await agent.run(query)
    print(result)
    # Code executes in Azure/OpenAI sandbox, not locally

# Run agent
import asyncio
asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop